# Train YOLOv8 — Lesion Detection
Runs on **Kaggle**. Expects the prepared YOLO dataset to be available under the working directory
(uploaded as a Kaggle dataset from the local `data_preparation.ipynb` output).

---
## 0. Environment setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# Copy src and prepared YOLO dataset into /kaggle/working
import shutil, sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input/datasets/nguyenquynghia")
WORKING_DIR  = Path("/kaggle/working")

# src
src_working = WORKING_DIR / "src"
if src_working.exists():
    shutil.rmtree(src_working)
shutil.copytree(KAGGLE_INPUT / "src-training/src", src_working)
if str(src_working) not in sys.path:
    sys.path.insert(0, str(src_working))
print(f"src → {src_working}")

# YOLO prepared dataset  (dataset/yolo)
dataset_dest = WORKING_DIR / "dataset" / "yolo"
if dataset_dest.exists():
    shutil.rmtree(dataset_dest)
dataset_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(KAGGLE_INPUT / "isic-2018-prepared/dataset/yolo", dataset_dest)
print(f"dataset/yolo → {dataset_dest}")

---
## 1. Train YOLOv8

In [ ]:
from yolo.train import train as train_yolo

model = train_yolo(epochs=100, batch=16, img_size=640)

---
## 2. Validate

In [ ]:
from yolo.train import validate as validate_yolo

validate_yolo()

---
## 3. Visualize — Ground Truth vs Prediction

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from pathlib import Path
from yolo.config import YOLO_OUTPUT, YOLO_DIR
from ultralytics import YOLO

weights_path = str(YOLO_OUTPUT / "lesion_detect" / "weights" / "best.pt")
model = YOLO(weights_path)

val_dir = YOLO_DIR / "images" / "val"
sample_img = sorted(val_dir.glob("*.jpg"))[0]
print(f"Sample: {sample_img.name}")

img = np.array(Image.open(sample_img).convert("RGB"))
H, W = img.shape[:2]

label_path = YOLO_DIR / "labels" / "val" / sample_img.with_suffix(".txt").name
gt_boxes = []
if label_path.exists():
    for line in label_path.read_text().strip().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        gt_boxes.append(((cx - bw/2)*W, (cy - bh/2)*H, (cx + bw/2)*W, (cy + bh/2)*H))

results   = model.predict(source=str(sample_img), imgsz=640, conf=0.25, verbose=False)
pred_boxes = results[0].boxes.xyxy.cpu().numpy()
pred_confs = results[0].boxes.conf.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, title in zip(axes, ["Ground Truth", "Prediction"]):
    ax.imshow(img); ax.set_title(f"{sample_img.name}\n{title}"); ax.axis("off")

for (x1, y1, x2, y2) in gt_boxes:
    axes[0].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=2, edgecolor="lime", facecolor="none"))
    axes[0].text(x1, y1-4, "lesion (GT)", color="lime", fontsize=9, bbox=dict(facecolor="black", alpha=0.4))

for (x1, y1, x2, y2), conf in zip(pred_boxes, pred_confs):
    axes[1].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=2, edgecolor="red", facecolor="none"))
    axes[1].text(x1, y1-4, f"{conf:.2f}", color="red", fontsize=9, bbox=dict(facecolor="black", alpha=0.4))

plt.tight_layout(); plt.show()
print(f"GT: {len(gt_boxes)}  Predicted: {len(pred_boxes)}")

---
## 4. Save weights

In [ ]:
import zipfile
from pathlib import Path

output_dir = Path("/kaggle/working/outputs/yolo")
output_zip = "/kaggle/working/yolo_weights.zip"

if output_dir.exists():
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in output_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to("/kaggle/working"))
                print(f"  {f.relative_to('/kaggle/working')}")
    print(f"\nSaved → {output_zip}  (download from Kaggle Output tab)")
else:
    print("No YOLO output directory found.")